# Generate topobathy GeoTIFF from GEBCO

Download a regional subset from [GEBCO 2025](https://www.gebco.net/data-products/gridded-bathymetry-data/gebco2025-grid) (CEDA OPeNDAP) at **native grid resolution** (~400 m), save a GeoTIFF.

Use this notebook when you do not have local bathymetry yet.

Downstream use:

1. Point `paths.aoi_file` and `paths.topobathy_file` in the case JSON to the correspondent paths.
2. Continue with `02_Create_mesh_by_bluemesh2d.ipynb`.

Requires network access for the GEBCO download.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr
from shapely.geometry import Polygon
from utils.config import PATHS, case_name, resolve

## Parameters

Two AoI modes are supported:

1. **Use existing AoI shapefile** from `paths.aoi_file`
2. **Create AoI shapefile from vertices** (entered clockwise)

In both cases, GEBCO bounds are derived automatically as AoI bounds + 50 km buffer.

In [ ]:
# Padding around AoI bounds for GEBCO subset
buffer_km = 50.0

# AoI mode:
# - True  -> read existing shapefile from paths.aoi_file
# - False -> create shapefile from vertices_clockwise below
use_existing_aoi = True

# Required only when use_existing_aoi = False.
# Enter lon/lat vertices in clockwise order and close the polygon by repeating
# the first vertex at the end (left-to-right progression around the AoI).
vertices_clockwise = [
    # (-171.95, -13.78),
    # (-171.55, -13.78),
    # (-171.55, -14.15),
    # (-171.95, -14.15),
    # (-171.95, -13.78),
]

# GEBCO 2025 OPeNDAP endpoint (ice-surface / land elevation)
gebco_url = (
    "https://dap.ceda.ac.uk/thredds/dodsC/bodc/gebco/global/gebco_2025/"
    "ice_surface_elevation/netcdf/GEBCO_2025.nc"
)

In [ ]:
# Inputs/outputs
input_aoi = resolve(PATHS["aoi_file"])
output_tiff = resolve(f"inputs/GEBCO/{case_name()}_GEBCO.tiff")

output_tiff.parent.mkdir(parents=True, exist_ok=True)

print(f"AoI shp  : {input_aoi}")
print(f"GeoTIFF  : {output_tiff}")
print(f"buffer km: {buffer_km}")

## Build/Load AoI and derive buffered bounds

- If `use_existing_aoi=True`, read `paths.aoi_file`.
- If `use_existing_aoi=False`, create `paths.aoi_file` from `vertices_clockwise`.

Then expand outer bounds by 50 km (in projected meters) before subsetting GEBCO.

In [ ]:
if use_existing_aoi:
    if not input_aoi.exists():
        raise FileNotFoundError(
            f"AoI file not found: {input_aoi}\n"
            "Set use_existing_aoi=False to create it from vertices_clockwise."
        )
    aoi_gdf = gpd.read_file(input_aoi).to_crs("EPSG:4326")
    print(f"Loaded existing AoI: {input_aoi}")
else:
    if len(vertices_clockwise) < 4:
        raise ValueError(
            "vertices_clockwise must contain at least 4 points (including closure)."
        )

    polygon = Polygon(vertices_clockwise)
    if not polygon.is_valid:
        raise ValueError("Provided vertices_clockwise do not form a valid polygon.")

    if not polygon.exterior.is_ccw:
        # As requested: vertices should be clockwise.
        pass
    else:
        print("Warning: vertices are counter-clockwise; proceeding anyway.")

    input_aoi.parent.mkdir(parents=True, exist_ok=True)
    aoi_gdf = gpd.GeoDataFrame({"name": ["aoi"]}, geometry=[polygon], crs="EPSG:4326")
    aoi_gdf.to_file(input_aoi)
    print(f"Created AoI shapefile: {input_aoi}")

# Buffer in meters using Web Mercator, then return to lon/lat.
buffered = aoi_gdf.to_crs("EPSG:3857").buffer(buffer_km * 1000.0).to_crs("EPSG:4326")

bbox = dict(zip(["lon_min", "lat_min", "lon_max", "lat_max"], buffered.total_bounds))
print({k: round(v, 5) for k, v in bbox.items()})
aoi_gdf

## Download and subset GEBCO

In [ ]:
GEBCO = xr.open_dataset(gebco_url)
GEBCO

In [ ]:
elev_name = "elevation" if "elevation" in GEBCO else list(GEBCO.data_vars)[0]
print(f"elevation variable: {elev_name}")

# GEBCO latitude is usually descending — slice high to low
lat_slice = slice(bbox["lat_min"], bbox["lat_max"])
lon_slice = slice(bbox["lon_min"], bbox["lon_max"])

gebco_sel = GEBCO.sel(lon=lon_slice, lat=lat_slice)
gebco_sel

## Write GeoTIFF (native GEBCO resolution)

In [ ]:
gebco_raster = (
    gebco_sel[elev_name]
    .rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    .rio.write_crs("EPSG:4326", inplace=False)
)

gebco_raster.rio.to_raster(output_tiff)
print(f"Wrote {output_tiff}")
print(f"CRS: {gebco_raster.rio.crs}")
print(f"Resolution (deg): {gebco_raster.rio.resolution()}")
print(f"Shape: {gebco_raster.shape}")

## Quick preview

In [ ]:
topobathy = rioxarray.open_rasterio(output_tiff).squeeze(drop=True)

fig, ax = plt.subplots(figsize=(9, 6))
topobathy.plot.imshow(ax=ax, vmin=-200, vmax=0, cmap="terrain")
aoi_gdf.boundary.plot(ax=ax, color="black", linewidth=1.2)
ax.set_title(output_tiff.name)
fig.tight_layout()
plt.show()

## Next step

For a new case, make sure your case config has:

```json
"paths": {
    "aoi_file": "inputs/AoI_Polygons/<Case>_AoI.shp",
    "topobathy_file": "inputs/GEBCO/<Case>_GEBCO.tiff"
}
```

Workflow options in this notebook:
- **Existing AoI**: keep `use_existing_aoi=True` and point `aoi_file` to your `.shp`.
- **No AoI yet**: set `use_existing_aoi=False`, fill `vertices_clockwise`, and run to create the `.shp` automatically.

Then run `02_Create_mesh_by_bluemesh2d.ipynb`.